# Delta Lake 查询示例

本 Notebook 提供 Delta Lake 数据湖的交互式查询示例。

In [ ]:
# 初始化 Spark Session
import sys
sys.path.insert(0, '/home/jovyan')

from configs.spark_config import (
    get_spark_session, read_delta_table,
    ODS_PATH, DWD_PATH, DWS_PATH, ADS_PATH
)

spark = get_spark_session('Jupyter_DeltaQuery')
print('Spark Session 初始化成功')

## 1. 列出所有表

In [ ]:
# 列出 Delta Lake 中的所有表
tables = spark.sql('SHOW TABLES IN delta.`s3a://financial-data-lake/ods`')
tables.show()

## 2. 查询 ODS 层数据

In [ ]:
# 查询客户表
df_customer = spark.sql("""
    SELECT * 
    FROM delta.`s3a://financial-data-lake/ods/ods_customer`
    LIMIT 10
""")
df_customer.show()

## 3. 查询 DWS 层汇总数据

In [ ]:
# 查询客户价值分布
df_value = spark.sql("""
    SELECT 
        level_name,
        COUNT(DISTINCT customer_id) as customer_count,
        SUM(total_aum) as total_aum
    FROM delta.`s3a://financial-data-lake/dws/dws_customer_value_profile`
    WHERE stat_date = '2026-06-23'
    GROUP BY level_name
    ORDER BY total_aum DESC
""")
df_value.show()

## 4. 查询 ADS 层应用数据

In [ ]:
# 查询管理驾驶舱 KPI
df_dashboard = spark.sql("""
    SELECT 
        stat_date,
        total_customers,
        total_aum,
        active_customers,
        aum_change
    FROM delta.`s3a://financial-data-lake/ads/ads_executive_dashboard`
    ORDER BY stat_date DESC
    LIMIT 7
""")
df_dashboard.show()

## 5. Delta Lake Time Travel（时间旅行）

Delta Lake 支持查询历史版本数据。

In [ ]:
# 查看表的历史版本
history = spark.sql("""
    DESCRIBE HISTORY delta.`s3a://financial-data-lake/dws/dws_customer_value_profile`
""")
history.show()

In [ ]:
# 查询指定时间点的数据
# df_history = spark.sql("""
#     SELECT * 
#     FROM delta.`s3a://financial-data-lake/dws/dws_customer_value_profile`
#     TIMESTAMP AS OF '2026-06-20'
# """)
# df_history.show()

## 6. 自定义查询

在下方单元格中编写你的 SQL 查询。

In [ ]:
# 在这里编写你的查询
# 示例：
# df = spark.sql("""
#     SELECT 
#         
#     FROM delta.`s3a://financial-data-lake/ods/ods_transaction`
#     WHERE 
# """)
# df.show()